In [287]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from arch import arch_model
from data_wrangling import *
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler
from black_scholes import *

## Pulling data from various data files

In [2]:
# data_dxy = yf.download("DX-Y.NYB", start="2000-01-01", end="2024-01-01", threads=False)
# data_2yr = yf.download("^IRX", start="2000-01-01", end="2024-01-01", threads=False)
# data_10yr = yf.download("^TNX", start="2000-01-01", end="2024-01-01", threads=False)
# data_vix = yf.download("^VIX", start="2000-01-01", end="2024-01-01", threads=False)

data_dxy = pd.read_csv("data/data_dxy.csv", index_col=0).Close
data_dxy.index = pd.to_datetime(data_dxy.index)
data_2yr = pd.read_csv("data/data_2yr.csv", index_col=0).Close
data_10yr = pd.read_csv("data/data_10yr.csv", index_col=0).Close
data_vix = pd.read_csv("data/data_vix.csv", index_col=0).Close
data_vix.index = pd.to_datetime(data_vix.index)

data_2s10s = data_2yr - data_10yr
data_2s10s.index = pd.to_datetime(data_2s10s.index)

aapl_time_series = pd.read_hdf("data/all_tickers_time_series.hf5", key="AAPL").drop_duplicates().set_index("date")
aapl_returns = aapl_time_series.prc.pct_change()

In [3]:
print(len(aapl_time_series.index))
print(len(data_dxy.index))
print(len(data_2s10s.index))
print(len(data_vix.index))

6037
6064
6031
6037


Apparantly these dataframes have a difference in data lengths

In [4]:
dates_aapl = set(aapl_time_series.index)
dates_dxy = set(data_dxy.index)
dates_2s10s = set(data_2s10s.index)
dates_vix = set(data_vix.index)

common_dates = dates_aapl & dates_dxy & dates_2s10s & dates_vix
not_common_dates = (dates_aapl | dates_dxy | dates_2s10s | dates_vix) - common_dates

print(len(common_dates))
print(len(not_common_dates))

6030
36


In [5]:
sorted(list(not_common_dates))

[Timestamp('2000-01-17 00:00:00'),
 Timestamp('2000-02-21 00:00:00'),
 Timestamp('2000-07-04 00:00:00'),
 Timestamp('2000-11-23 00:00:00'),
 Timestamp('2001-01-15 00:00:00'),
 Timestamp('2001-02-19 00:00:00'),
 Timestamp('2001-07-04 00:00:00'),
 Timestamp('2001-09-03 00:00:00'),
 Timestamp('2001-11-22 00:00:00'),
 Timestamp('2002-01-21 00:00:00'),
 Timestamp('2002-02-18 00:00:00'),
 Timestamp('2002-05-27 00:00:00'),
 Timestamp('2002-07-04 00:00:00'),
 Timestamp('2002-11-28 00:00:00'),
 Timestamp('2003-07-04 00:00:00'),
 Timestamp('2003-09-01 00:00:00'),
 Timestamp('2003-11-11 00:00:00'),
 Timestamp('2004-01-19 00:00:00'),
 Timestamp('2004-02-16 00:00:00'),
 Timestamp('2004-06-11 00:00:00'),
 Timestamp('2004-07-05 00:00:00'),
 Timestamp('2004-09-06 00:00:00'),
 Timestamp('2004-11-25 00:00:00'),
 Timestamp('2004-12-24 00:00:00'),
 Timestamp('2005-01-17 00:00:00'),
 Timestamp('2005-02-21 00:00:00'),
 Timestamp('2005-07-04 00:00:00'),
 Timestamp('2005-10-10 00:00:00'),
 Timestamp('2005-11-

Turns out these dates are the days where it is a banking holiday, thus stock market not opened, but various indexes continues to count <br>
Note, 2016-10-10 and 2016-11-11 are banking holidays, but AAPL continued to trade

In [6]:
new_aapl = aapl_time_series[aapl_time_series.index.isin(common_dates)]
new_data_dxy = data_dxy[data_dxy.index.isin(common_dates)]
new_data_2s10s = data_2s10s[data_2s10s.index.isin(common_dates)]
new_data_vix = data_vix[data_vix.index.isin(common_dates)]

## EDA

In [7]:
adf_test(new_data_dxy)
print('---')
adf_test(new_data_2s10s)
print('---')
adf_test(new_data_vix)

ADF Statistic: -1.5625279264735055
p-value: 0.5023845733881521
Non-stationary: Consider differencing or other transformations
---
ADF Statistic: -1.4974466159694706
p-value: 0.5347740811956415
Non-stationary: Consider differencing or other transformations
---
ADF Statistic: -5.809785756316987
p-value: 4.429021238909754e-07
Stationary: No differencing required


4.429021238909754e-07

In [8]:
new_data_dxy_d1 = new_data_dxy.diff().dropna()
adf_test(new_data_dxy_d1)
print("---")
new_data_2s10s_d1 = new_data_2s10s.diff().dropna()
adf_test(new_data_2s10s_d1)

ADF Statistic: -23.419312260383563
p-value: 0.0
Stationary: No differencing required
---
ADF Statistic: -15.039411414105626
p-value: 9.642058296564496e-28
Stationary: No differencing required


9.642058296564496e-28

Datasets to be used: <br>
DXY: new_data_dxy_d1 <br>
2s10s: new_data_2s10s_d1 <br>
VIX: new_data_vix

## Testing DXY

In [9]:
goldfeld_quandt_test(new_aapl['prc'].pct_change().dropna(), new_data_dxy.pct_change().dropna())

{'F-statistic': 0.6113312480717363, 'p-value': 0.9999999999999999}

Based on GQ test, since p-value is close to 1, we do not reject H0, suggesting that homoscedasticity, meeting the assumption of constant variance

In [10]:
white_test(new_aapl['prc'].pct_change().dropna(), new_data_dxy.pct_change().dropna())

{'LM-statistic': 1.3307228729956266,
 'LM-test p-value': 0.514087684071219,
 'F-statistic': 0.6651771741276042,
 'F-test p-value': 0.5142201696270757}

Based on White's test, failed to reject H0 as both LM-statistic and F-statistics are both not significant. This suggests that variance of residual is homoscedastic

## Testing 2s10s

In [11]:
aapl_returns = new_aapl['prc'].pct_change().dropna()
dxy_returns = new_data_2s10s.pct_change().dropna()

merged_data = pd.merge(aapl_returns, dxy_returns, left_index=True, right_index=True, how='inner')

merged_data = merged_data.replace([np.inf, -np.inf], np.nan).dropna()

gq_test_results = goldfeld_quandt_test(merged_data.iloc[:, 0], merged_data.iloc[:, 1])
white_test_results = white_test(merged_data.iloc[:, 0], merged_data.iloc[:, 1])

print(gq_test_results)
print(white_test_results)

{'F-statistic': 3.4617351398397833, 'p-value': 4.929946278060511e-240}
{'LM-statistic': 0.2703809463549116, 'LM-test p-value': 0.8735495080903555, 'F-statistic': 0.13512925303093257, 'F-test p-value': 0.8736056361016313}


GQ test and White's test showing conflicting results (GQ suggesting heteroscedasticity while White's showing homoscedasticity). White's test generally considered stricter and more comprehensive as it makes few assumptions of form of heteroscedasticity, thus we will follow White's results

## Testing VIX

In [12]:
goldfeld_quandt_test(new_aapl['prc'].pct_change().dropna(), new_data_vix.pct_change().dropna())

{'F-statistic': 1.632306755824517, 'p-value': 3.974481603250593e-41}

In [13]:
white_test(new_aapl['prc'].pct_change().dropna(), new_data_vix.pct_change().dropna())

{'LM-statistic': 746.6320908035916,
 'LM-test p-value': 7.42850595868739e-163,
 'F-statistic': 425.87008861589266,
 'F-test p-value': 1.0084858502142215e-173}

Both GQ and White suggests presence of heteroscedasticity by rejecting null hypothesis of homoscedasticity.

## Regressing Exogenous Variables on Stock Price Returns

In [14]:
dxy_returns = new_data_dxy_d1.pct_change().dropna()
d_2s10s_returns = new_data_2s10s_d1.pct_change().dropna()
vix_returns = new_data_vix.pct_change().dropna()
# inf_dates = d_2s10s_returns[d_2s10s_returns==np.inf].index
# zero_dates = d_2s10s_returns[d_2s10s_returns==0].index
# print(f"dates with np.inf: {inf_dates}, dates with zeros: {zero_dates}")

Note that the inf comes from measuring a percentage change from a 0 value to a non-zero value, and 0 comes from measuring a percentage change from a non-zero value to a 0 value <br>
Perhaps, best way to move forward is to drop these dates

In [15]:
# X = sm.add_constant(pd.concat([dxy_returns, d_2s10s_returns, vix_returns], axis=1)).drop(inf_dates).drop(zero_dates)
# y = new_aapl['prc'].pct_change().dropna().drop(inf_dates).drop(zero_dates)

X_df = pd.concat([dxy_returns, d_2s10s_returns, vix_returns], axis=1)
comb_df = X_df.merge(new_aapl['prc'].pct_change(), left_index=True, right_index=True, how="inner")
comb_df.columns = ["DXY", "2s10s", "VIX", "returns"]
comb_df.replace([np.inf, -np.inf], np.nan, inplace=True)
comb_df.dropna(inplace=True)
comb_df

X = comb_df[["DXY", "2s10s", "VIX"]]
y = comb_df["returns"]
ols_model = sm.OLS(y, X).fit()
residuals = ols_model.resid

## Building the GARCH Model

$$
\sigma_t^2 = \alpha_0 + \alpha_1 \epsilon_{t-1}^2 + \beta_1 \sigma_{t-1}^2
$$

The above is the general form of GARCH(1,1) model

In [16]:
garch = arch_model(residuals, vol="Garch", p=1, q=1)
garch_fit = garch.fit()
garch_fit.summary()

Iteration:      1,   Func. Count:      6,   Neg. LLF: 770230567.7775779
Iteration:      2,   Func. Count:     17,   Neg. LLF: -13071.45399295337
Optimization terminated successfully    (Exit mode 0)
            Current function value: -13071.453974124563
            Iterations: 6
            Function evaluations: 17
            Gradient evaluations: 2


c:\Users\Gavin\anaconda3\Lib\site-packages\arch\univariate\base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.00076. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                   None   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                13071.5
Distribution:                  Normal   AIC:                          -26134.9
Method:            Maximum Likelihood   BIC:                          -26108.2
                                        No. Observations:                 5923
Date:                Thu, Oct 10 2024   Df Residuals:                     5922
Time:                        00:40:39   Df Model:                            1
                                  Mean Model                                 
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
mu         9.2735e-04  7.878e-04      1.177      0.239 [-6.167e-04,2.471e-03]
                               Volatility Model                              
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
omega      7.6004e-05  5.013e-05      1.516      0.129 [-2.224e-05,1.742e-04]
alpha[1]       0.0500  3.909e-02      1.279      0.201   [-2.662e-02,  0.127]
beta[1]        0.8500  8.535e-02      9.959  2.311e-23      [  0.683,  1.017]
=============================================================================

Covariance estimator: robust
"""

In [17]:
vol_forecast = garch_fit.conditional_volatility

## Incorporating Exogenous Variables in GARCH Model

In [18]:
garch_x_model = arch_model(comb_df["returns"], vol="Garch", p=1, q=1, x=comb_df[["DXY", "2s10s", "VIX"]])
garch_x_fit = garch_x_model.fit()
garch_x_fit.summary()

c:\Users\Gavin\anaconda3\Lib\site-packages\arch\univariate\base.py:311: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0008707. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(


Iteration:      1,   Func. Count:      6,   Neg. LLF: 164827800.8043421
Iteration:      2,   Func. Count:     18,   Neg. LLF: 32099.2587436947
Iteration:      3,   Func. Count:     30,   Neg. LLF: 981169651760.2346
Iteration:      4,   Func. Count:     44,   Neg. LLF: 3138941.714535622
Iteration:      5,   Func. Count:     57,   Neg. LLF: 33411380395.449257
Iteration:      6,   Func. Count:     69,   Neg. LLF: -12669.853446815352
Optimization terminated successfully    (Exit mode 0)
            Current function value: -12669.853430317416
            Iterations: 10
            Function evaluations: 69
            Gradient evaluations: 6


<class 'statsmodels.iolib.summary.Summary'>
"""
                     Constant Mean - GARCH Model Results                      
==============================================================================
Dep. Variable:                returns   R-squared:                       0.000
Mean Model:             Constant Mean   Adj. R-squared:                  0.000
Vol Model:                      GARCH   Log-Likelihood:                12669.9
Distribution:                  Normal   AIC:                          -25331.7
Method:            Maximum Likelihood   BIC:                          -25305.0
                                        No. Observations:                 5923
Date:                Thu, Oct 10 2024   Df Residuals:                     5922
Time:                        00:40:40   Df Model:                            1
                                  Mean Model                                 
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
mu         6.1878e-04  7.773e-04      0.796      0.426 [-9.047e-04,2.142e-03]
                               Volatility Model                              
=============================================================================
                 coef    std err          t      P>|t|       95.0% Conf. Int.
-----------------------------------------------------------------------------
omega      8.7070e-05  7.096e-05      1.227      0.220 [-5.200e-05,2.261e-04]
alpha[1]       0.0500  3.346e-02      1.494      0.135   [-1.558e-02,  0.116]
beta[1]        0.8500  9.580e-02      8.873  7.122e-19      [  0.662,  1.038]
=============================================================================

Covariance estimator: robust
"""

## Forecasting

In [19]:
forecast_horizon = 5
forecast = garch_x_fit.forecast(horizon=forecast_horizon)
variance_forecast = forecast.variance[-1:]
variance_forecast.apply(np.sqrt)

,h.1,h.2,h.3,h.4,h.5
2023-12-29,0.024384,0.024944,0.025437,0.025873,0.026259


## OOP Portion

In [283]:
class DataProcessing:
    def __init__(self, y_variable_ticker: str):
        self.data_dxy = pd.read_csv("data/data_dxy.csv", index_col=0).Close
        self.data_dxy.index = pd.to_datetime(self.data_dxy.index)

        self.data_2yr = pd.read_csv("data/data_2yr.csv", index_col=0).Close
        self.data_10yr = pd.read_csv("data/data_10yr.csv", index_col=0).Close
        self.data_2s10s = self.data_2yr - self.data_10yr
        self.data_2s10s.index = pd.to_datetime(self.data_2s10s.index)

        self.data_vix = pd.read_csv("data/data_vix.csv", index_col=0).Close
        self.data_vix.index = pd.to_datetime(self.data_vix.index)

        self.ticker_time_series = pd.read_hdf("data/all_tickers_time_series.hf5", key=y_variable_ticker).drop_duplicates().set_index("date")
        self.ticker_returns = self.ticker_time_series.prc.pct_change().dropna()

    def get_common_dates(self):
        dates_y = set(self.ticker_returns.index)
        dates_dxy = set(self.data_dxy.index)
        dates_2s10s = set(self.data_2s10s.index)
        dates_vix = set(self.data_vix.index)

        common_dates = dates_y & dates_dxy & dates_2s10s & dates_vix
        not_common_dates = (dates_y | dates_dxy | dates_2s10s | dates_vix) - common_dates

        return common_dates, not_common_dates
    
    def get_common_data(self):
        common_dates, _ = self.get_common_dates()

        new_y = self.ticker_returns[self.ticker_returns.index.isin(common_dates)]
        new_data_dxy = self.data_dxy[self.data_dxy.index.isin(common_dates)]
        new_data_2s10s = self.data_2s10s[self.data_2s10s.index.isin(common_dates)]
        new_data_vix = self.data_vix[self.data_vix.index.isin(common_dates)]

        return new_y, new_data_dxy, new_data_2s10s, new_data_vix
    
    def get_stationary_data(self):
        """
        Returns [y, dxy, 2s10s, vix] data that has been differenced until it is stationary
        """
        def difference_until_stationary(data):
            result = adf_test(data)
            differenced = 0

            while result > 0.05:
                data = data.diff().dropna()
                result = adf_test(data)
                differenced += 1

                if differenced > 2:
                    raise ValueError("Data could not be made stationary after 2 differences")
            return data, result
        
        stationary_results = []
        count = 0
        for data in self.get_common_data():
            try:
                count += 1
                if count == 1:
                    stationary_data = data
                else:
                    stationary_data, result = difference_until_stationary(data)
                stationary_results.append(stationary_data)
            except ValueError as e:
                print(f"Error with differencing for {data}: {e}")
        
        return stationary_results
    
    def data_split(self):
        y_stationary, dxy_stationary, d_2s10s_stationary, vix_stationary = self.get_stationary_data()
        X_df = pd.concat([dxy_stationary, d_2s10s_stationary, vix_stationary], axis=1)
        comb_df = X_df.merge(y_stationary, left_index=True, right_index=True, how="inner")
        comb_df.columns = ["DXY", "2s10s", "VIX", "returns"]
        comb_df.replace([np.inf, -np.inf], np.nan, inplace=True)
        comb_df.dropna(inplace=True)
        comb_df = comb_df.sort_index()
        X = comb_df[["DXY", "2s10s", "VIX"]]
        y = comb_df["returns"]
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        y_scaled = scaler.fit_transform(y.values.reshape(-1, 1))
        X_scaled = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
        y_scaled = pd.Series(y_scaled.flatten(), index=y.index)
        X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_scaled, test_size=0.3, random_state=42, shuffle=False)
        X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42, shuffle=False)
        return X_train, X_test, y_train, y_test, X_val, y_val

class GARCHX:
    def __init__(self, y_variable_ticker: str, X_train, X_test, y_train, y_test, X_val, y_val, p=1, q=1):
        self.ticker = y_variable_ticker
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.X_val = X_val
        self.y_val = y_val
        self.p = p
        self.q = q

    def train_get_GARCHX_fit(self):
        """
        Appropriate if exogenous variables have an impact on the volatility of the return and the returns itself.

        Exogeneous variables are included in the volatility equation, allowing them to influence variance of returns.
        """
        garch_x_model = arch_model(self.y_train, vol="Garch", p=self.p, q=self.q, x=self.X_train, dist="t")
        self.garch_x_fit = garch_x_model.fit(disp="off")
    
    def train_GARCHX_fit_summary(self):
        return self.garch_x_fit.summary()
    

    def get_rolling_predictions(self):
        self.rolling_prediction = []
        rolling_y_train = self.y_train
        rolling_x_train = self.X_train
        for i in range(len(self.y_test)):
            rolling_y_train = pd.concat([self.y_train, self.y_test[:i+1]])
            rolling_x_train = pd.concat([self.X_train, self.X_test[:i+1]])
            garch_x_model = arch_model(rolling_y_train, vol="Garch", p=self.p, q=self.q, x=rolling_x_train, dist="t")
            garch_x_fit = garch_x_model.fit(disp="off")
            forecast = garch_x_fit.forecast(horizon=1)
            forecast_variance = forecast.variance.values[-1]
            self.rolling_prediction.append(forecast_variance)        

    def evaluate_rmse(self):
        squared_errors = (self.y_test**2).values - self.rolling_prediction
        rmse = np.sqrt(np.mean(squared_errors**2))
        return "RMSE (squared residuals vs predicted variance):", rmse
    
    def evaluate_directional_accuracy(self):
        realized_volatility = (self.y_test ** 2).values
        predicted_variance = np.array(self.rolling_prediction).flatten()

        realized_direction = np.sign(np.diff(realized_volatility))
        predicted_direction = np.sign(np.diff(predicted_variance))

        correct_directions = np.sum(realized_direction == predicted_direction)
        total_directions = len(realized_direction)

        directional_accuracy = correct_directions / total_directions
        directional_accuracy = directional_accuracy * 100
        return "Directional Accuracy:%", directional_accuracy


    def evaluate_hit_rate(self):
        realized_volatility = (self.y_test ** 2).values

        predicted_variance = np.array(self.rolling_prediction)

        high_threshold = np.percentile(realized_volatility, 75)
        low_threshold = np.percentile(realized_volatility, 25)

        realized_high = realized_volatility >= high_threshold
        predicted_high = predicted_variance >= high_threshold

        realized_low = realized_volatility <= low_threshold
        predicted_low = predicted_variance <= low_threshold

        high_hit_rate = np.mean(realized_high == predicted_high)
        low_hit_rate = np.mean(realized_low == predicted_low)

        return f"High Volatility Hit Rate: {high_hit_rate * 100:.2f}%, Low Volatility Hit Rate: {low_hit_rate * 100:.2f}%"
    
    def evaluate_qlike(self):
        realized_volatility = (self.y_test ** 2).values
        predicted_variance = np.array(self.rolling_prediction).flatten()
        qlike = np.mean(realized_volatility / predicted_variance - np.log(realized_volatility / predicted_variance) - 1)
        return "QLIKE:", qlike


class GARCHXModelPipeline:
    def __init__(self, tickers_list, X_train, X_test, y_train, y_test, X_val, y_val, p=1, q=1):
        self.tickers_list = tickers_list
        self.p = p
        self.q = q
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.X_val = X_val
        self.y_val = y_val
        self.metric_df = pd.DataFrame()

    def build_evaluate(self):
        for ticker in self.tickers_list:
            garchx = GARCHX(ticker, self.X_train, self.X_test, self.y_train, self.y_test, self.X_val, self.y_val, self.p, self.q)
            garchx.train_get_GARCHX_fit()
            garchx.get_rolling_predictions()

            rmse = garchx.evaluate_rmse()
            directional_accuracy = garchx.evaluate_directional_accuracy()
            qlike = garchx.evaluate_qlike()

            self.metric_df = self.metric_df.append({
                "Ticker": ticker,
                "RMSE": rmse[1],
                "Directional Accuracy": directional_accuracy[1],
                "QLIKE": qlike[1]
            }, ignore_index=True)
        return self.metric_df
    
    def get_best_performing(self, n=10):
        best_performing = self.metric_df.sort_values(by="RMSE").head(n)
        return best_performing
    
class TradingStrategy:
    def __init__(self, selected_models, X_train, X_test, y_train, y_test, X_val, y_val):
        self.selected_models = selected_models
        self.X_train = X_train
        self.X_test = X_test
        self.y_train = y_train
        self.y_test = y_test
        self.X_val = X_val
        self.y_val = y_val

    
    def get_implied_volatility(self, date: str):
        tickers = self.selected_models["Ticker"].tolist()
        implied_volatility = {}
        for ticker in tickers:
            price_data = pd.read_hdf("data/all_tickers_time_series.hf5", key=ticker).drop_duplicates().set_index("date")
            price_data = price_data.loc[date]
            price = price_data["bidlo"]

            options_data = pd.read_hdf("data/all_options_data.h5", key=ticker)
            options_data = options_data.loc[date]
            options_data = options_data[options_data["exdate"] == (pd.to_datetime(date) + pd.DateOffset(days=4))]
            options_data = options_data[options_data["strike"]/1000 == round(price)]
            c_option_data = options_data[options_data["cp_flag"] == "C"]
            p_option_data = options_data[options_data["cp_flag"] == "P"]

            c_implied_volatility = implied_volatility(S=price, 
                                                      K=c_option_data["strike"]/1000,
                                                      T=4/252,
                                                      r=0.01,
                                                      type="call",
                                                      market_price=c_option_data["best_bid"])
            p_implied_volatility = implied_volatility(S=price,
                                                      K=p_option_data["strike"]/1000,
                                                      T=4/252,
                                                      r=0.01,
                                                      type="put",
                                                      market_price=p_option_data["best_offer"])
            avg_implied_volatility = (c_implied_volatility + p_implied_volatility) / 2
            implied_volatility[ticker] = avg_implied_volatility
        return implied_volatility


    def weekly_forecast(self):
        forecast_data = {}
        for index, row in self.selected_models.iterrows():
            ticker = row["Ticker"]
            garchx = GARCHX(ticker, self.X_train, self.X_test, self.y_train, self.y_test, self.X_val, self.y_val)
            garchx.train_get_GARCHX_fit()
            forecast = garchx.garch_x_fit.forecast(horizon=5).variance.values[-1]
            forecast_ema = pd.Series(forecast).ewm(span=5).mean().values[-1]

            forecast_data[ticker] = forecast_ema
        return forecast_data
    
    def trade(self):
        

# class GARCH:
#     def __init__(self, y_variable_ticker: str):
#         self.data_process = DataProcessing(y_variable_ticker)
#         self.X_train, self.X_test, self.y_train, self.y_test = self.data_process.data_split()

#     def train_get_OLS_residuals(self):
#         self.ols_model = sm.OLS(self.y_train, self.X_train).fit()
#         residuals = self.ols_model.resid
#         return residuals
    
#     def train_get_GARCH_fit(self, p=1, q=1):
#         """
#         More appropriate for for exogenous variables only affecting the mean (returns), 
#         and after accounting for them, model the volatility of the unexplained part of the returns.

#         Approach assumes volatility process itself is independent of the exogenous variables.
#         """
#         residuals = self.train_get_OLS_residuals()
#         garch = arch_model(residuals, vol="Garch", p=p, q=q)
#         self.garch_fit = garch.fit()

#     def train_GARCH_fit_summary(self):
#         return self.garch_fit.summary()
    
    
#     def test_get_GARCH_forecast_rmse(self, p=1, q=1):
#         test_residuals = self.y_test - self.ols_model.predict(self.X_test)
#         garch_forecast = self.garch_fit.forecast(horizon=len(self.y_test))
#         garch_vol_forecast = np.sqrt(garch_forecast.variance[-len(self.y_test):])
#         rmse = calculate_rmse(test_residuals, garch_vol_forecast.T)
#         return rmse

In [284]:
garchx = GARCHX("NVDA")

garchx.train_get_GARCHX_fit()
# garchx.train_GARCHX_fit_summary()
garchx.get_rolling_predictions()

ADF Statistic: -1.5532218556161643
p-value: 0.5070384434257603
Non-stationary: Consider differencing or other transformations
ADF Statistic: -23.404801432198557
p-value: 0.0
Stationary: No differencing required
ADF Statistic: -1.5019807350724121
p-value: 0.532531271013979
Non-stationary: Consider differencing or other transformations
ADF Statistic: -15.046030924037936
p-value: 9.441687562067877e-28
Stationary: No differencing required
ADF Statistic: -5.809842828092417
p-value: 4.4277355586878604e-07
Stationary: No differencing required


In [286]:
garchx.evaluate_rmse()[1]

10.444863149939339

In [278]:
garchx.evaluate_directional_accuracy()

Realized volatility shape: (1206,)
Predicted variance shape: (1206,)
Realized direction shape: (1205,)
Predicted direction shape: (1205,)


'Directional Accuracy: 67.30%'

In [279]:
garchx.evaluate_hit_rate()

'High Volatility Hit Rate: 51.28%, Low Volatility Hit Rate: 74.96%'

In [280]:
garchx.evaluate_qlike()

'QLIKE: 1.5416052740882402'

In [282]:
tickers_df = pd.read_excel("data/RIY Index Constituents.xlsx")
unique_tickers = set()
cols = tickers_df.columns
for col in cols:
    tickers_for_year = tickers_df[col].unique()
    for ticker in tickers_for_year:
        unique_tickers.add(ticker.split()[0].strip().upper())

for ticker in unique_tickers:
    try:
        garchx = GARCHX(ticker)
        garchx.train_get_GARCHX_fit()
        garchx.get_rolling_predictions()
        print(f"{ticker}: {garchx.evaluate_rmse()}")
        print(f"{ticker}: {garchx.evaluate_directional_accuracy()}")
        print(f"{ticker}: {garchx.evaluate_hit_rate()}")
        print(f"{ticker}: {garchx.evaluate_qlike()}")
    except Exception as e:
        print(f"Error with {ticker}: {e}")

{'RYAN',
 'EXPE',
 'HUBB',
 'OXY',
 'TOL',
 'UNM',
 'PARA',
 'ALB',
 'SNX',
 'EXP',
 'SPOT',
 'JBL',
 'PR',
 'CDW',
 'HPQ',
 'NKE',
 'ADSK',
 'ALK',
 'CIVI',
 'BAC',
 'AMP',
 'FE',
 'SMG',
 'BLK',
 'GRAL',
 'RGLD',
 'CNA',
 'ILMN',
 'MGM',
 'KO',
 'MDB',
 'NTRA',
 'MOS',
 'AMH',
 'CMI',
 'SLGN',
 'TRU',
 'AON',
 'VSTS',
 'ETSY',
 'CERT',
 'GPK',
 'CRUS',
 'KDP',
 'LBRDA',
 'ETR',
 'MDLZ',
 'SJM',
 'HRB',
 'NNN',
 'PNR',
 'ZTS',
 'OZK',
 'NTAP',
 'NVST',
 'BOKF',
 'BDX',
 'CRWD',
 'TT',
 'PG',
 'DIS',
 'STT',
 'KMX',
 'WFRD',
 'CTAS',
 'AAON',
 'PPL',
 'CXT',
 'AXP',
 'UTHR',
 'INSP',
 'CRM',
 'RYN',
 'HEI',
 'HII',
 'EIX',
 'PTC',
 'EHC',
 'ARE',
 'FLO',
 'PPC',
 'CSX',
 'ACI',
 'OSK',
 'PK',
 'STAG',
 'WTM',
 'LOW',
 'COIN',
 'ANSS',
 'QS',
 'GILD',
 'TECH',
 'MMM',
 'HOLX',
 'TOST',
 'LLYVA',
 'LII',
 'BBY',
 'CSCO',
 'AMZN',
 'HLT',
 'USB',
 'J',
 'MP',
 'LVS',
 'PANW',
 'PINC',
 'SQ',
 'WPC',
 'EBAY',
 'HOOD',
 'KRC',
 'ROST',
 'TDY',
 'GLOB',
 'L',
 'UHAL',
 'COO',
 'VKTX',
 'WWD'

In [315]:
date = "2021-01-04"

price_data = pd.read_hdf("data/all_tickers_time_series.hf5", key="AAPL").drop_duplicates().set_index("date")
price = price_data.loc[date]["bidlo"]
price

126.76

In [317]:
options_data = pd.read_hdf("data/all_options_data.h5", key="AAPL").set_index("date")
options_data = options_data.loc[date]
options_data = options_data[options_data["exdate"] == (pd.to_datetime(date) + pd.DateOffset(days=4))]
options_data = options_data[options_data["strike_price"]/1000 == round(price)]
options_data

,ticker,exdate,cp_flag,strike_price,best_bid,best_offer,open_interest,impl_volatility,delta,gamma,theta,vega,volume
date,,,,,,,,,,,,,
2021-01-04,AAPL,2021-01-08,C,127000.0,3.70,3.80,1981.0,0.438951,0.667158,0.061089,-98.6419,4.922918,7097.0
2021-01-04,AAPL,2021-01-08,P,127000.0,1.45,1.47,8555.0,0.463496,-0.340291,0.058356,-104.9291,4.964450,17546.0
